# 3-3 pandas から直接グラフを描く

3-1、3-2 では `plt.bar(...)` `plt.plot(...)` のように、**matplotlib に x と y を渡す書き方** をしてきました。

実は pandas には **「DataFrame や Series そのものから `.plot()` を呼ぶ」** ショートカットがあります。これを使うと、3-1, 3-2 の内容が **さらに短く** 書けます。

## このノートのゴール

- **`Series.plot()`** / **`DataFrame.plot()`** で 1〜2 行でグラフが描ける
- **`kind="bar"` `"line"` `"scatter"` `"pie"`** で種類を切り替え
- **複数列を持つ DataFrame** から直接、複数系列の折れ線・棒グラフが出る
- **`stacked=True`** で積み上げ棒グラフ
- 3-1 / 3-2 の **`plt.xxx` 流** と、3-3 の **`df.plot` 流** をいつ使い分けるか分かる

## なぜ pandas から直接描くのか

3-1 では棒グラフをこう書きました:

```python
plt.bar(by_product.index, by_product.values)
```

pandas 流ならこうです:

```python
by_product.plot.bar()
```

**index と values を分けて渡す手間がなくなります**。複数系列のときは、3-2 では `for` ループで何本も `plot` を呼びましたが、pandas 流なら **DataFrame をそのまま渡すだけで凡例付きの複数系列** になります。

「**素早く描いて中身を確かめたい**」場面で重宝します。

## 準備

3-1, 3-2 と同じ準備です。

In [ ]:
!pip install -q japanize-matplotlib

import matplotlib.pyplot as plt
import japanize_matplotlib
import pandas as pd

from google.colab import drive
drive.mount('/content/drive')

BASE = "/content/drive/MyDrive/python-data-basics"
DATA = f"{BASE}/data/pandas"

df = pd.read_excel(f"{DATA}/sales_2026-01.xlsx", sheet_name="売上", parse_dates=["date"])
df.head()

## 1. Series から直接描く — `.plot()`

`groupby` の結果（Series）の **末尾に `.plot(...)` を足すだけ** でグラフになります。

`Series.plot()` は **既定で折れ線**。棒なら `.plot.bar()`、横棒なら `.plot.barh()` です。

In [ ]:
daily = df.groupby("date")["amount"].sum()

# 折れ線（既定）
daily.plot(figsize=(10, 4), title="日次売上推移", marker="o")
plt.ylabel("売上 (円)")
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
by_product = df.groupby("product_name")["amount"].sum().sort_values(ascending=False)

# 棒グラフ
by_product.plot.bar(figsize=(9, 4), title="商品別 売上合計", color="steelblue")
plt.ylabel("売上 (円)")
plt.show()

**ポイント**: `.plot.bar()` と書くと、**x 軸ラベルは自動で index、棒の高さは自動で values** になります。3-1 で `plt.bar(by_product.index, by_product.values)` と書いていた手間が消えます。

> `title` や `figsize` は `.plot(...)` の引数で直接渡せます。`ylabel` や `grid` などは従来通り `plt.xxx` で追加します。

## 2. `kind` 引数で種類を切り替え

上の `daily.plot()` `by_product.plot.bar()` は、**`kind="..."`** という形でも書けます。意味は同じです。

| 書き方A | 書き方B (kind 指定) | 種類 |
|---|---|---|
| `.plot()` | `.plot(kind="line")` | 折れ線 |
| `.plot.bar()` | `.plot(kind="bar")` | 縦棒 |
| `.plot.barh()` | `.plot(kind="barh")` | 横棒 |
| `.plot.scatter(x=..., y=...)` | `.plot(kind="scatter", x=..., y=...)` | 散布図 |
| `.plot.pie()` | `.plot(kind="pie")` | 円グラフ |
| `.plot.hist()` | `.plot(kind="hist")` | ヒストグラム |

**どちらの書き方でも OK**。コードを書くときは好みで選んでください。

In [ ]:
# 売上構成比を円グラフで（kind="pie" の例）
by_product.plot.pie(figsize=(6, 6), title="商品別 売上構成比", autopct="%.1f%%")
plt.ylabel("")   # 円グラフの場合は y ラベルを空に
plt.show()

## 3. 複数列の DataFrame から直接 — 凡例付きの複数系列が一発

3-2 で書いた **「商品ごとに `for` で `plot` を呼ぶ」** ループは、pandas 流なら **書く必要がなくなります**。

コツは、データを **「行: 日付、列: 商品、値: 売上」** の **横長 (wide)** の形に変換すること。Excel で言う **ピボットテーブル** です。

pandas では **`pivot_table()`** を使います。

In [ ]:
# 行: 日付、列: 商品、値: 売上合計 という形に変換
pivot = df.pivot_table(
    index="date",
    columns="product_name",
    values="amount",
    aggfunc="sum",
    fill_value=0,    # その日にその商品が売れなかったセルは 0 で埋める
)
pivot.head()

In [ ]:
# この pivot をそのまま .plot() に渡すと、列ごとに線が引かれ、凡例も自動で付く
pivot.plot(figsize=(12, 5), title="商品別 日次売上推移", marker="o")
plt.ylabel("売上 (円)")
plt.grid(True, alpha=0.3)
plt.show()

**たった 2 行** で、3-2 で `for` ループを書いた図と同じものが出ます。

ピボットして DataFrame.plot に流す、この流れは第4章の **月次レポート自動生成** で何度も使うことになります。

## 4. 積み上げ棒グラフ — `stacked=True`

`pivot` の形のままで、**`.plot.bar(stacked=True)`** にすると **積み上げ棒グラフ** になります。

「**日ごとの売上を、商品の内訳まで含めて見たい**」というときの定番です。

In [ ]:
pivot.plot.bar(stacked=True, figsize=(14, 5), title="日次売上の商品別内訳")
plt.ylabel("売上 (円)")
plt.legend(loc="upper left", bbox_to_anchor=(1, 1))   # 凡例をグラフの外に出す
plt.show()

> 💡 `plt.legend(loc="upper left", bbox_to_anchor=(1, 1))` は **凡例をグラフの右外** に出すおまじない。商品数が多くて中に入れると邪魔なときに使います。

## 5. `plt.xxx` 流 と `df.plot` 流 の使い分け

| 場面 | おすすめ | 理由 |
|---|---|---|
| **ざっとデータを見たい** | `df.plot()` 流 | 圧倒的に短い |
| **複数系列を一気に** | `pivot.plot()` 流 | for ループ不要 |
| **積み上げ棒** | `pivot.plot.bar(stacked=True)` 流 | 1行 |
| **レポート用に細かく整える** | 両方 (`df.plot` → 追加で `plt.xxx`) | 表示後に `plt.title` `plt.ylabel` などを追加できる |
| **完全に独自レイアウト** | `plt.xxx` 流 | 細かい制御が効く |

**`df.plot()` 流で 8 割書けて、残りを `plt.xxx` で仕上げる** のが現実的な書き方です。

## 練習問題

本文のコードを少しだけ変える形の演習です。`df` `daily` `by_product` `pivot` はそのまま使ってください。

1. **`by_product` を横棒** で描いてください。本文 `.plot.bar()` の **`bar` を `barh` に変える** だけ。
2. **日付別の取引件数** を `.plot()` で折れ線にしてください。本文の `daily = df.groupby("date")["amount"].sum()` の **`.sum()` を `.count()` に変える** だけ。
3. **`pivot`** を **積み上げ棒** で描いてください。本文 `pivot.plot.bar(stacked=True, ...)` をそのまま実行するだけで OK。タイトルを `"商品内訳 (1月)"` に変えてください。

In [ ]:
# ここに書いてみてください



<details>
<summary>答えを見る</summary>

```python
# 1. by_product を横棒で
by_product.plot.barh(figsize=(8, 4), title="商品別 売上合計", color="steelblue")
plt.xlabel("売上 (円)")
plt.show()

# 2. 日付別の取引件数
daily_count = df.groupby("date")["amount"].count()
daily_count.plot(figsize=(10, 4), title="日次 取引件数", marker="o")
plt.ylabel("件数")
plt.grid(True, alpha=0.3)
plt.show()

# 3. pivot を積み上げ棒で
pivot.plot.bar(stacked=True, figsize=(14, 5), title="商品内訳 (1月)")
plt.ylabel("売上 (円)")
plt.legend(loc="upper left", bbox_to_anchor=(1, 1))
plt.show()
```

</details>

## よくあるエラー

### 1. `df.plot()` でグラフが出ない
→ `import matplotlib.pyplot as plt` が抜けている。pandas は内部で matplotlib を使うので、matplotlib 側の import も必要。

### 2. `pivot_table` を作っても列がたくさん出てしまう
→ `columns=` に指定した列の **ユニークな値の数** だけ列が増える。ここでは商品が 6 つあるので 6 列でOK。多すぎる場合は、`df` の段階で絞り込んでから pivot する。

### 3. 棒グラフの x ラベルが斜めにならない
→ `.plot.bar()` は **既定で斜め表示**。逆に水平に戻したいときは `plt.xticks(rotation=0)`。

### 4. 凡例がグラフに被って読めない
→ `plt.legend(loc="upper left", bbox_to_anchor=(1, 1))` で外に出す。

### 5. `.plot.pie()` で円グラフが横長になる
→ `figsize=(6, 6)` のように **縦横同じ値** にして正方形にする。

## まとめ

- **`Series.plot()` / `DataFrame.plot()`** で 1 行グラフ
- `kind="line"` `"bar"` `"barh"` `"scatter"` `"pie"` `"hist"` で種類切替（`.plot.bar()` のように dot 記法も OK）
- **`pivot_table`** で wide format に変換 → **そのまま `.plot()`** で複数系列
- **`stacked=True`** で積み上げ棒
- `df.plot` 流で素早く描き、足りない部分を `plt.xxx` で補うのが実務での書き方

次は **3-4「体裁を整える」** で、フォント・色・軸の数値書式（3桁区切りや「万円」表示）など、**レポートに耐える見た目** に仕上げる方法を扱います。